# 03 — Modeling: Train/Test Split

Starts from `data/processed/crop_loss_model_ready.csv` (the clean, final
output of `02_feature_exploration.ipynb` — 12 columns, 72,762 rows, 0
missing values, 0 duplicates, `household_id` still included and the data
still whole). This notebook's first job: split it into train/test sets
without leaking a household across both sides.

In [1]:
import pandas as pd

# household_id explicitly as str -- see CLAUDE.md's float-precision gotcha.
df_model = pd.read_csv("../data/processed/crop_loss_model_ready.csv", dtype={"household_id": str})
print(df_model.shape)
df_model.head()

(72762, 12)


,household_id,crop_name,survey_year,household_size,region_code,is_rural,region_name,rainfall_belg_mm,rainfall_belg_pct_of_avg,rainfall_meher_mm,rainfall_meher_pct_of_avg,loss_occurred
0,1010101601002,MAIZE,2011,8.0,1.0,1.0,Tigray,120.648096,112.0,643.046036,95.0,0
1,1010101601002,MAIZE,2013,9.0,1.0,1.0,Tigray,72.747426,67.5,610.313750,90.2,0
2,1010101601002,MAIZE,2015,8.0,1.0,1.0,Tigray,111.048456,103.1,555.302842,82.0,0
3,1010101601002,MILLET,2011,8.0,1.0,1.0,Tigray,120.648096,112.0,643.046036,95.0,0
4,1010101601002,MILLET,2013,9.0,1.0,1.0,Tigray,72.747426,67.5,610.313750,90.2,0


## Train/test split — grouped by `household_id`

A household can report multiple crops, so a naive random split could put
the same household's rows on both sides of train/test, letting a model
partly recognize a specific household instead of learning a generalizable
pattern. `GroupShuffleSplit` (not plain `train_test_split`, which has no
`groups` parameter) keeps every row from a given household entirely on one
side. Verified below: zero `household_id` overlap, and the loss rate in
each split stays close to the full dataset's rate.

In [2]:
from sklearn.model_selection import GroupShuffleSplit

splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(splitter.split(df_model, groups=df_model["household_id"]))

train_df = df_model.iloc[train_idx].copy()
test_df = df_model.iloc[test_idx].copy()

overlap = set(train_df["household_id"]) & set(test_df["household_id"])

print("train:", train_df.shape, "| test:", test_df.shape)
print("household_id overlap between train/test:", len(overlap))
print("train loss rate:", train_df["loss_occurred"].mean().round(3))
print("test loss rate:", test_df["loss_occurred"].mean().round(3))

train: (58343, 12) | test: (14419, 12)
household_id overlap between train/test: 0
train loss rate: 0.067
test loss rate: 0.068


## Encode features

Building `X`/`y` from an explicit whitelist rather than dropping columns —
safer, since `household_id` (grouping key) and `region_name` (display label,
duplicate of `region_code`) can never leak into the model even if
`df_model` gains columns later.

- **Categorical** (`crop_name`, `region_code`, `survey_year`): one-hot
  encoded. `crop_name` has ~188 categories, but one-hot is still the right
  default here — it doesn't impose a false ordering the way integer-coding
  would, and with 58k+ training rows the extra width is fine. The encoder is
  **fit on train only** and applied to test with `handle_unknown="ignore"`
  (any category test has that train never saw becomes all-zeros, rather than
  erroring or leaking test-set category information back into the encoding).
- **Numeric** (`household_size`, `is_rural`, the 4 rainfall columns): used
  as-is, no scaling — tree-based models (the planned direction) don't need
  it, and it's simple to add later if a linear model wants it.

In [3]:
from sklearn.preprocessing import OneHotEncoder

CATEGORICAL_COLS = ["crop_name", "region_code", "survey_year"]
NUMERIC_COLS = ["household_size", "is_rural", "rainfall_belg_mm", "rainfall_belg_pct_of_avg",
                "rainfall_meher_mm", "rainfall_meher_pct_of_avg"]
TARGET_COL = "loss_occurred"

encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
encoder.fit(train_df[CATEGORICAL_COLS])

train_encoded = pd.DataFrame(
    encoder.transform(train_df[CATEGORICAL_COLS]),
    columns=encoder.get_feature_names_out(CATEGORICAL_COLS),
    index=train_df.index,
)
test_encoded = pd.DataFrame(
    encoder.transform(test_df[CATEGORICAL_COLS]),
    columns=encoder.get_feature_names_out(CATEGORICAL_COLS),
    index=test_df.index,
)

X_train = pd.concat([train_df[NUMERIC_COLS], train_encoded], axis=1)
X_test = pd.concat([test_df[NUMERIC_COLS], test_encoded], axis=1)
y_train = train_df[TARGET_COL]
y_test = test_df[TARGET_COL]

print("X_train:", X_train.shape, "| X_test:", X_test.shape)
print("y_train loss rate:", y_train.mean().round(3), "| y_test loss rate:", y_test.mean().round(3))
print("\ncategories per encoded column (first 10):")
print(list(encoder.get_feature_names_out(CATEGORICAL_COLS))[:10])

X_train: (58343, 144) | X_test: (14419, 144)
y_train loss rate: 0.067 | y_test loss rate: 0.068

categories per encoded column (first 10):
['crop_name_AMBOSHIKA', 'crop_name_APPLES', 'crop_name_AVOCADOS', 'crop_name_BANANAS', 'crop_name_BARLEY', 'crop_name_BEER ROOT', 'crop_name_BEERROOT', 'crop_name_BLACK CUMIN', 'crop_name_BLACK PEPPER', 'crop_name_BLACKCUMIN']


## Sanity check — X/y ready for modeling

Confirms the encoding step didn't silently break anything before fitting
any model: no `NaN`s in `X_train`/`X_test`, every column fully numeric,
`X`/`y` row-aligned by index on both sides, and `X_train`/`X_test` have
identical columns (so a model trained on one can score the other).

In [4]:
print("X_train NaN total:", X_train.isna().sum().sum())
print("X_test NaN total:", X_test.isna().sum().sum())
print("X_train non-numeric cols:", list(X_train.select_dtypes(exclude="number").columns))
print("X_test non-numeric cols:", list(X_test.select_dtypes(exclude="number").columns))
print("X_train/y_train index match:", X_train.index.equals(y_train.index))
print("X_test/y_test index match:", X_test.index.equals(y_test.index))
print("columns identical train vs test:", list(X_train.columns) == list(X_test.columns))

X_train NaN total: 0
X_test NaN total: 0
X_train non-numeric cols: []
X_test non-numeric cols: []
X_train/y_train index match: True
X_test/y_test index match: True
columns identical train vs test: True
